# Customer Churn Prediction with Machine Learning

## Project Overview

Customer churn is a major challenge for telecommunications companies. Predicting which customers are likely to leave can help organizations design targeted retention strategies and reduce customer attrition.

In this project, several machine learning classification algorithms are developed and compared to predict customer churn using customer demographic, service, contract and billing information.

The main models evaluated are:

- Logistic Regression
- Random Forest
- XGBoost

Model performance is evaluated using ROC-AUC as the primary metric, together with PR-AUC, precision, recall and F1-score.

The analysis also includes cross-validation, hyperparameter optimization, probability threshold optimization and SHAP-based model interpretability.

## Objectives

1. Understand the main characteristics of the customer dataset.
2. Identify variables associated with customer churn.
3. Build and compare several classification models.
4. Optimize the best-performing model.
5. Evaluate its performance on an untouched test set.
6. Optimize the classification threshold according to predictive performance.
7. Interpret the model and identify the main drivers of customer churn.

# 1. Environment and Reproducibility

This section defines the libraries, random seed and general configuration used throughout the analysis.

A fixed random seed is used to ensure that the experiments are reproducible.

In [1]:
%pip install -q kagglehub xgboost shap

Note: you may need to restart the kernel to use updated packages.


In [6]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
import shap

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_validate,
    cross_val_predict,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 4983

np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

# 2. Dataset

The dataset used in this project is the Telco Customer Churn dataset.

It contains information about telecommunications customers, including demographic characteristics, subscribed services, contract information, tenure and billing information.

The target variable is `Churn`, which indicates whether a customer left the company.

path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

print(f"Dataset downloaded to: {path}")

In [10]:
path = kagglehub.dataset_download(
    "blastchar/telco-customer-churn"
)

print("Dataset downloaded successfully.")
print(f"Path: {path}")

csv_files = list(Path(path).rglob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file in csv_files:

    print(f" - {file.name}")



Dataset downloaded successfully.
Path: /Users/alejandromaduenogalvez/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1
CSV files found: 1
 - WA_Fn-UseC_-Telco-Customer-Churn.csv


In [11]:
DATA_FILE = csv_files[0]

df = pd.read_csv(DATA_FILE)

print(f"Dataset shape: {df.shape}")

df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2.1 Dataset Dimensions

Before performing any preprocessing, we inspect the number of observations and variables in the dataset.

In [12]:
df.info()
df.describe(include="all").T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.2 Variable Description

The main variables in the dataset describe customer demographics, services, contracts and billing characteristics.

| Variable | Description |
|---|---|
| `customerID` | Unique customer identifier |
| `gender` | Customer gender |
| `SeniorCitizen` | Whether the customer is a senior citizen |
| `Partner` | Whether the customer has a partner |
| `Dependents` | Whether the customer has dependents |
| `tenure` | Number of months the customer has been with the company |
| `PhoneService` | Whether the customer has phone service |
| `InternetService` | Type of internet service |
| `Contract` | Customer contract type |
| `PaymentMethod` | Customer payment method |
| `MonthlyCharges` | Monthly customer charges |
| `TotalCharges` | Total customer charges |
| `Churn` | Whether the customer churned |

# 3. Data Quality

Before modelling, the dataset is inspected for missing values, duplicated observations, incorrect data types and other potential data quality issues.

In [ ]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percentage": df.isna().mean() * 100
})

missing.sort_values("missing", ascending=False)

In [ ]:
empty_total_charges = df[
    df["TotalCharges"].astype(str).str.strip() == ""
]

print(
    f"Empty TotalCharges values: "
    f"{len(empty_total_charges)}"
)

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Number of duplicated rows: {duplicate_count}")

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)
print(

    f"Missing TotalCharges after conversion: "

    f"{df['TotalCharges'].isna().sum()}"

)

In [ ]:
def data_quality_report(data):
    report = pd.DataFrame({
        "dtype": data.dtypes,
        "missing": data.isna().sum(),
        "missing_pct": data.isna().mean() * 100,
        "unique": data.nunique()
    })

    return report.sort_values(
        "missing_pct",
        ascending=False
    )


data_quality_report(df)

# 4. Exploratory Data Analysis

## 4.1 Target Variable

The first step of the exploratory analysis is to examine the distribution of the target variable.

Understanding the class distribution is particularly important because customer churn is not evenly distributed across the two classes.

In [ ]:
target_counts = df["Churn"].value_counts()

target_percentages = (
    df["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

target_summary

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Churn",
    hue="Churn",
    legend=False
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

## 4.2 Numerical Variables

The main numerical variables are:

- `tenure`
- `MonthlyCharges`
- `TotalCharges`

Their distributions are examined both globally and according to churn status.

In [ ]:
numeric_features = [

    "tenure",

    "MonthlyCharges",

    "TotalCharges"

]

df[numeric_features].describe().T

In [ ]:
for feature in numeric_features:

    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=df,
        x=feature,
        hue="Churn",
        kde=True,
        element="step",
        stat="density",
        common_norm=False
    )

    plt.title(f"{feature} Distribution by Churn")
    plt.tight_layout()
    plt.show()

In [ ]:
for feature in numeric_features:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="Churn",
        y=feature,
        hue="Churn",
        legend=False
    )

    plt.title(f"{feature} by Churn Status")
    plt.tight_layout()
    plt.show()

## 4.3 Categorical Variables

Categorical variables are examined in relation to customer churn.

Rather than only examining the frequency of each category, churn rates are calculated within each category. This provides a more meaningful interpretation of the relationship between customer characteristics and churn.

In [ ]:
for feature in categorical_features:

    churn_rate = (
        df.groupby(feature, observed=True)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .sort_values(ascending=False)
    )

    print(f"\n{feature}")
    display(
        churn_rate
        .rename("Churn Rate (%)")
        .to_frame()
    )

In [ ]:
selected_categorical = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport"
]

for feature in selected_categorical:

    plt.figure(figsize=(9, 5))

    sns.barplot(
        data=df,
        x=feature,
        y=(df["Churn"] == "Yes").astype(int),
        errorbar=None
    )

    plt.title(f"Churn Rate by {feature}")
    plt.ylabel("Churn Rate")
    plt.xlabel(feature)

    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.show()

## 4.4 Numerical Correlations

Correlation analysis is used to identify potential relationships among numerical variables.

This analysis is descriptive and is not used to select variables before cross-validation.

In [ ]:
corr = df[numeric_features].corr()

plt.figure(figsize=(7, 5))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")

plt.tight_layout()
plt.show()

# 5. Feature Definition

The customer identifier is removed because it is an identifier rather than a predictive characteristic.

The target variable is encoded as:

- `0` → No churn
- `1` → Churn

In [ ]:
X = df.drop(
    columns=["Churn", "customerID"]
)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print(f"Features: {X.shape[1]}")
print(f"Observations: {X.shape[0]}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

# 7. Preprocessing Pipeline

Preprocessing is implemented using a scikit-learn `ColumnTransformer` and `Pipeline`.

This ensures that preprocessing operations are learned only from the training data within each cross-validation fold, preventing data leakage.

Numerical variables are median-imputed and standardized.

Categorical variables are imputed using the most frequent category and one-hot encoded.

In [ ]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

In [ ]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

# 8. Baseline Model

Before fitting machine learning models, a Dummy Classifier is used as a baseline.

This provides a reference point to determine whether the predictive models learn useful patterns beyond simply predicting the majority class.

In [ ]:
baseline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        DummyClassifier(
            strategy="most_frequent"
        )
    )
])

# 9. Cross-Validation Strategy

Model development is evaluated using stratified repeated cross-validation.

Five folds and three repetitions are used, resulting in 15 validation folds.

Stratification preserves the proportion of churned and non-churned customers in each fold.

In [ ]:
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE
)

In [ ]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

# 10. Model Development

## 10.1 Logistic Regression

Logistic Regression is used as an interpretable linear baseline for the classification problem.

In [ ]:
logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    )
])

## 10.2 Random Forest

Random Forest is a nonlinear ensemble method based on multiple decision trees.

In [ ]:
rf_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

## 10.3 XGBoost

XGBoost is a gradient boosting algorithm based on decision trees.

It is evaluated as the main high-performance candidate because of its ability to model nonlinear relationships and interactions between predictors.

In [ ]:
xgb_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

In [ ]:
models = {
    "Baseline": baseline,
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "PR-AUC": scores["test_pr_auc"].mean(),
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean()
    })

model_results = (
    pd.DataFrame(results)
    .sort_values(
        "ROC-AUC",
        ascending=False
    )
)

model_results

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=model_results,
    x="ROC-AUC",
    y="Model"
)

plt.title("Cross-Validated ROC-AUC")
plt.xlim(0.5, 0.9)

plt.tight_layout()
plt.show()

# 11. Hyperparameter Optimization

The best-performing models are further optimized using randomized hyperparameter search.

The optimization is performed exclusively on the training set using repeated cross-validation.

ROC-AUC is used as the optimization metric.

In [ ]:
xgb_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

In [ ]:
xgb_param_grid = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__min_child_weight": [1, 3, 5, 7],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "classifier__gamma": [0, 0.05, 0.1, 0.2],
    "classifier__reg_alpha": [0, 0.01, 0.1],
    "classifier__reg_lambda": [1, 5, 10, 20]
}

In [ ]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=50,
    scoring="roc_auc",
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    ),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(
    X_train,
    y_train
)

In [ ]:
print("Best ROC-AUC:")
print(xgb_search.best_score_)

print("\nBest parameters:")

for parameter, value in xgb_search.best_params_.items():
    print(f"{parameter}: {value}")

# 12. Final Model

The optimized XGBoost pipeline is selected as the final candidate model.

Importantly, the test set has not been used during model selection or hyperparameter optimization.

In [ ]:
final_model = xgb_search.best_estimator_

final_model.fit(
    X_train,
    y_train
)

In [ ]:
y_test_proba = final_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
test_roc_auc = roc_auc_score(
    y_test,
    y_test_proba
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

print(f"Test ROC-AUC: {test_roc_auc:.4f}")
print(f"Test PR-AUC:  {test_pr_auc:.4f}")

In [ ]:
y_test_pred_050 = (
    y_test_proba >= 0.50
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_050,
        target_names=["No Churn", "Churn"]
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_test_pred_050
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Threshold 0.50")

plt.tight_layout()
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_test_proba
)

plt.title("ROC Curve — Final XGBoost Model")
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(
    y_test,
    y_test_proba
)

plt.title(
    "Precision-Recall Curve — Final XGBoost Model"
)

plt.show()

# 13. Classification Threshold Optimization

The default classification threshold of 0.50 is not necessarily optimal for customer churn prediction.

Because identifying potential churners is important for retention strategies, different probability thresholds are evaluated.

The F1-score is used as the main criterion for threshold selection because it balances precision and recall.

In [ ]:
oof_proba = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    ),
    method="predict_proba",
    n_jobs=-1
)[:, 1]

In [ ]:
thresholds = np.arange(
    0.10,
    0.91,
    0.01
)

threshold_results = []

for threshold in thresholds:

    y_pred = (
        oof_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_train,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_train,
            y_pred,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results.sort_values(
    "f1",
    ascending=False
).head(10)

In [ ]:
best_threshold_row = (
    threshold_results
    .loc[
        threshold_results["f1"].idxmax()
    ]
)

optimal_threshold = (
    best_threshold_row["threshold"]
)

print(
    f"Optimal threshold: "
    f"{optimal_threshold:.2f}"
)

print(
    f"Precision: "
    f"{best_threshold_row['precision']:.4f}"
)

print(
    f"Recall: "
    f"{best_threshold_row['recall']:.4f}"
)

print(
    f"F1: "
    f"{best_threshold_row['f1']:.4f}"
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    label="Precision"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    label="Recall"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    label="F1"
)

plt.axvline(
    optimal_threshold,
    linestyle="--",
    label=f"Optimal = {optimal_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_test_pred_optimal = (
    y_test_proba >= optimal_threshold
).astype(int)

print(
    classification_report(
        y_test,
        y_test_pred_optimal,
        target_names=["No Churn", "Churn"]
    )
)

In [ ]:
threshold_comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1"
    ],
    "Threshold 0.50": [
        precision_score(
            y_test,
            y_test_pred_050
        ),
        recall_score(
            y_test,
            y_test_pred_050
        ),
        f1_score(
            y_test,
            y_test_pred_050
        )
    ],
    f"Threshold {optimal_threshold:.2f}": [
        precision_score(
            y_test,
            y_test_pred_optimal
        ),
        recall_score(
            y_test,
            y_test_pred_optimal
        ),
        f1_score(
            y_test,
            y_test_pred_optimal
        )
    ]
})

threshold_comparison

# 14. Model Interpretability

Predictive performance alone is not sufficient to understand customer churn.

SHAP (SHapley Additive exPlanations) is used to investigate how individual features contribute to the model's predictions.

The analysis focuses on global feature importance and the direction of feature effects.

In [ ]:
trained_preprocessor = (
    final_model
    .named_steps["preprocessor"]
)

trained_xgb = (
    final_model
    .named_steps["classifier"]
)

X_train_transformed = (
    trained_preprocessor
    .transform(X_train)
)

feature_names = (
    trained_preprocessor
    .get_feature_names_out()
)

In [ ]:
explainer = shap.TreeExplainer(
    trained_xgb
)

shap_values = explainer(
    X_train_transformed
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_train_transformed,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(
    shap_values,
    X_train_transformed,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)

plt.tight_layout()
plt.show()

# 15. Business Insights

The model provides several actionable insights into customer churn.

## Main Churn Drivers

The most influential variables identified by the model are discussed below based on the SHAP analysis.

### 1. [Feature]

[Interpretation based on the SHAP results.]

### 2. [Feature]

[Interpretation based on the SHAP results.]

### 3. [Feature]

[Interpretation based on the SHAP results.]

## Retention Implications

The predictions could be used to prioritize customers with high estimated churn probability for targeted retention campaigns.

The classification threshold can also be adjusted according to the relative costs of false positives and false negatives.

# 16. Final Results

The final XGBoost model is evaluated on the previously untouched test set.

The main performance metrics are:

- ROC-AUC
- PR-AUC
- Precision
- Recall
- F1-score

ROC-AUC measures the model's ability to rank churned customers above non-churned customers independently of a specific classification threshold.

The classification threshold is optimized separately when converting predicted probabilities into binary churn predictions.

In [ ]:
final_results = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1"
    ],
    "Test Performance": [
        roc_auc_score(
            y_test,
            y_test_proba
        ),
        average_precision_score(
            y_test,
            y_test_proba
        ),
        precision_score(
            y_test,
            y_test_pred_optimal
        ),
        recall_score(
            y_test,
            y_test_pred_optimal
        ),
        f1_score(
            y_test,
            y_test_pred_optimal
        )
    ]
})

final_results

# 17. Limitations and Future Work

Although the final model provides useful predictive performance, several limitations should be considered.

### Limitations

- The dataset represents a single telecommunications dataset and may not generalize to other companies or markets.
- The analysis is based on observational data and therefore does not establish causal relationships.
- The selected classification threshold depends on the assumed business objective.
- External validation on an independent dataset was not available.

### Future Work

Potential improvements include:

- Probability calibration.
- Cost-sensitive classification based on the economic cost of churn.
- Bayesian or more advanced hyperparameter optimization.
- Comparison with CatBoost and LightGBM.
- External validation using another customer dataset.
- Deployment as an API or interactive dashboard.

# 18. Conclusion

This project developed an end-to-end machine learning pipeline for customer churn prediction.

Several classification algorithms were compared using stratified repeated cross-validation. XGBoost was subsequently optimized through hyperparameter search and evaluated on an untouched test set.

In addition to ranking performance using ROC-AUC and PR-AUC, the classification threshold was optimized to obtain a better balance between precision and recall.

Finally, SHAP was used to interpret the model and identify the variables that contribute most strongly to churn predictions.

The resulting workflow provides not only a predictive model but also an interpretable framework that can support data-driven customer retention strategies.